# 37. HS4 Explanatory-Note Ablation Study

실험 36의 해설서 기반 성능 개선을 분해해 검증합니다. 동일한 195건·5개 HS4·29개 제품군과 Outer LOPO를 유지하고, 다음 네 모델을 비교합니다.

- A. Baseline: Product + Ruling RRF
- B. Direct only: Baseline + 호별 직접 근거
- C. Exclusion only: Baseline + 제외 규정 근거
- D. Direct + Exclusion: Baseline + 두 해설서 채널

각 모델의 가중치와 집계 방식은 Outer Fold의 학습 데이터 안에서 Inner LOPO로만 선택합니다. 1차 지표는 HS4 Macro-F1, 2차 지표는 Accuracy입니다. Google Drive는 사용하지 않습니다.

## 실험상 주의사항

- 네 모델은 같은 외부 평가 표본에서 비교합니다.
- A는 실험 34의 설정(top3_mean, top3_mean, RRF weight=1)을 고정합니다.
- B·C·D는 각자의 설정을 Inner LOPO에서 선택합니다.
- Inner Fold의 학습 데이터에 검증 라벨이 존재하지 않는 경우 해당 Fold는 튜닝 평가에서 제외하고, 유효 표본 수와 커버리지를 기록합니다.
- 해설서의 제외 규정 연결은 검색 신호이며, 법적 원산지 판정을 대신하지 않습니다.
- 해설서 채널의 기여도는 ablation 결과로 판단하되, Outer LOPO 결과를 최종 비교 기준으로 사용합니다.

## 0. 환경 설정

In [ ]:
!pip -q install -U sentence-transformers scipy kiwipiepy
import re, zipfile, requests
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
WORK=Path('/content/37_hs4_explanatory_note_ablation')
WORK.mkdir(parents=True,exist_ok=True)
BASE='https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/'
print(WORK)

## 1. 실험 29와 동일한 데이터 파이프라인

In [ ]:
src=requests.get(BASE+'29_discriminative_features_experiment.ipynb').json()
for i in [3,5,7]: exec(''.join(src['cells'][i]['source']),globals())
print('평가 표본:',len(eval_df),'| HS6:',len(labels),'| 제품군:',eval_df.product_family.nunique(),'| HS4:',eval_df.hs4.nunique())
assert len(eval_df)==195 and len(labels)==11 and eval_df.product_family.nunique()==29 and eval_df.hs4.nunique()==5
hs4_labels=sorted(eval_df.hs4.astype(str).unique())
print('HS4 labels:',hs4_labels)
display(eval_df.groupby('hs4').agg(n=('hs4','size'),families=('product_family','nunique')))

## 2. 해설서 근거 채널 구성

In [ ]:
kb=pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
for c in ['knowledge_id','hs_code','rule_type','document_text','source_sheet','source_page_or_line']:
    if c not in kb: kb[c]=''
    kb[c]=kb[c].astype(str).str.strip()
kb['hs_digits']=kb.hs_code.map(lambda x:re.sub(r'\D','',x))
def clean_kb_text(x):
    lines=[]
    for line in str(x).splitlines():
        if re.match(r'^(규칙_ID|HS코드|자료출처|류|적용범위)\s*:',line.strip()): continue
        lines.append(line.strip())
    return ' '.join(x for x in lines if x)
kb['embedding_text']=kb.document_text.map(clean_kb_text)
direct_rows=[]
exclusion_rows=[]
for _,r in kb.iterrows():
    typ=r.rule_type
    own=r.hs_digits[:4]
    if own in hs4_labels and r.embedding_text and '제외' not in typ:
        direct_rows.append({**r.to_dict(),'candidate_hs4':own,'channel':'direct'})
    if '제외' in typ and r.embedding_text:
        refs=sorted(set(re.findall(r'(?:제\s*)?(\d{4})\s*호',r.document_text)))
        for h in refs:
            if h[:4] in hs4_labels: exclusion_rows.append({**r.to_dict(),'candidate_hs4':h[:4],'channel':'exclusion'})
direct_docs=pd.DataFrame(direct_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
exclusion_docs=pd.DataFrame(exclusion_rows).drop_duplicates(['knowledge_id','candidate_hs4']).reset_index(drop=True)
print('KB:',len(kb),'| Direct:',len(direct_docs),'| Exclusion links:',len(exclusion_docs))
display(pd.concat([direct_docs.groupby('candidate_hs4').size().rename('direct_docs'),exclusion_docs.groupby('candidate_hs4').size().rename('exclusion_links')],axis=1).reindex(hs4_labels).fillna(0).astype(int))
display(exclusion_docs[['knowledge_id','candidate_hs4','rule_type','source_page_or_line','document_text']])

## 3. 임베딩 생성 및 캐시

In [ ]:
CACHE=WORK/'embeddings_37.npz'
if CACHE.exists():
    z=np.load(CACHE,allow_pickle=True)
    query_emb,product_emb,ruling_emb,kb_direct_emb,kb_exclusion_emb=z['query_emb'],z['product_emb'],z['ruling_emb'],z['kb_direct_emb'],z['kb_exclusion_emb']
else:
    sbert=SentenceTransformer('intfloat/multilingual-e5-base')
    def encode_query(x): return sbert.encode(['query: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    def encode_passage(x): return sbert.encode(['passage: '+str(v) for v in x],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
    query_emb=encode_query(eval_df.query_text.tolist())
    product_emb=encode_passage(eval_df.query_text.tolist())
    ruling_emb=encode_passage(eval_df.ruling_text.tolist())
    kb_direct_emb=encode_passage(direct_docs.embedding_text.tolist()) if len(direct_docs) else np.empty((0,768),dtype=float)
    kb_exclusion_emb=encode_passage(exclusion_docs.embedding_text.tolist()) if len(exclusion_docs) else np.empty((0,768),dtype=float)
    np.savez_compressed(CACHE,query_emb=query_emb,product_emb=product_emb,ruling_emb=ruling_emb,kb_direct_emb=kb_direct_emb,kb_exclusion_emb=kb_exclusion_emb)
print(query_emb.shape,product_emb.shape,ruling_emb.shape,kb_direct_emb.shape,kb_exclusion_emb.shape)

## 4. HS4 순위 함수 및 검색 캐시

In [ ]:
def aggregate_hs4(scores,codes,method='top3_mean'):
    d=pd.DataFrame({'hs4':np.asarray(codes).astype(str),'score':np.asarray(scores,dtype=float)})
    if method=='max': out=d.groupby('hs4').score.max()
    elif method=='mean': out=d.groupby('hs4').score.mean()
    else: out=d.groupby('hs4').score.apply(lambda x:x.nlargest(min(3,len(x))).mean())
    return out.sort_values(ascending=False)
def rrf_two(a,b,k=60,w=1.0):
    pa={str(c):i+1 for i,c in enumerate(a.index)}
    pb={str(c):i+1 for i,c in enumerate(b.index)}
    return pd.Series({c:1/(k+pa.get(c,99999))+w/(k+pb.get(c,99999)) for c in hs4_labels}).sort_values(ascending=False)
def rank_kb(scores,docs,method):
    vals={}
    for c in hs4_labels:
        x=np.asarray(scores)[docs.candidate_hs4.astype(str).to_numpy()==c]
        if len(x)==0: vals[c]=-1.0
        elif method=='max': vals[c]=float(np.max(x))
        else: vals[c]=float(np.mean(np.sort(x)[-min(3,len(x)):]))
    return pd.Series(vals).sort_values(ascending=False)
def baseline_rank(train_idx,q):
    codes=eval_df.iloc[train_idx].hs4.astype(str).to_numpy()
    rp=aggregate_hs4(product_emb[train_idx]@query_emb[q],codes,'top3_mean')
    rr=aggregate_hs4(ruling_emb[train_idx]@query_emb[q],codes,'top3_mean')
    return rrf_two(rp,rr,w=1.0)
direct_codes=direct_docs.candidate_hs4.astype(str).to_numpy()
exclusion_codes=exclusion_docs.candidate_hs4.astype(str).to_numpy()
METHODS=['max','top3_mean']
WEIGHTS=[0.0,0.5,1.0,2.0]
direct_rank_cache={}
exclusion_rank_cache={}
for q in range(len(eval_df)):
    ds=kb_direct_emb@query_emb[q] if len(direct_docs) else np.array([])
    es=kb_exclusion_emb@query_emb[q] if len(exclusion_docs) else np.array([])
    direct_rank_cache[q]={m:rank_kb(ds,direct_docs,m) for m in METHODS}
    exclusion_rank_cache[q]={m:rank_kb(es,exclusion_docs,m) for m in METHODS}
splits=list(LeaveOneGroupOut().split(eval_df,eval_df.hs4,groups=eval_df.product_family))
print('Outer LOPO:',len(splits))

## 5. Inner LOPO 튜닝 및 Outer LOPO 예측

B는 Direct 가중치만, C는 Exclusion 가중치만, D는 두 채널의 가중치를 각각 선택합니다. 세 모델 모두 동일한 Inner validation 행을 사용하도록 Fold 유효성 조건을 공유합니다.

In [ ]:
FOLD_START=1
FOLD_END=5
def rrf_ablation(base,direct=None,exclusion=None,wd=0.0,we=0.0,k=60):
    ranks=[base]
    weights=[1.0]
    if direct is not None: ranks.append(direct); weights.append(wd)
    if exclusion is not None: ranks.append(exclusion); weights.append(we)
    pos=[{str(c):i+1 for i,c in enumerate(r.index)} for r in ranks]
    return pd.Series({c:sum(w/(k+p.get(c,99999)) for w,p in zip(weights,pos)) for c in hs4_labels}).sort_values(ascending=False)
CONFIGS_B=[(m,w) for m in METHODS for w in WEIGHTS]
CONFIGS_C=[(m,w) for m in METHODS for w in WEIGHTS]
CONFIGS_D=[(md,me,wd,we) for md in METHODS for me in METHODS for wd in WEIGHTS for we in WEIGHTS]
def predict_variant(base,q,variant,cfg):
    if variant=='A': rank=base
    elif variant=='B':
        m,w=cfg; rank=rrf_ablation(base,direct_rank_cache[q][m],None,w,0)
    elif variant=='C':
        m,w=cfg; rank=rrf_ablation(base,None,exclusion_rank_cache[q][m],0,w)
    else:
        md,me,wd,we=cfg; rank=rrf_ablation(base,direct_rank_cache[q][md],exclusion_rank_cache[q][me],wd,we)
    return str(rank.index[0])
def tune_variant(outer_train,variant):
    inner_df=eval_df.iloc[outer_train].reset_index(drop=True)
    inner_splits=list(LeaveOneGroupOut().split(inner_df,inner_df.hs4,groups=inner_df.product_family))
    configs={'B':CONFIGS_B,'C':CONFIGS_C,'D':CONFIGS_D}[variant]
    records={c:[] for c in configs}
    used=0
    for itr,iva in inner_splits:
        gtr=outer_train[itr]; gva=outer_train[iva]
        train_labels=set(eval_df.iloc[gtr].hs4.astype(str))
        valid_labels=set(eval_df.iloc[gva].hs4.astype(str))
        if not valid_labels.issubset(train_labels): continue
        used+=1
        for q in gva:
            base=baseline_rank(gtr,q)
            true=str(eval_df.iloc[q].hs4)
            for cfg in configs: records[cfg].append((true,predict_variant(base,q,variant,cfg)))
    if used==0: raise ValueError('유효한 Inner LOPO Fold가 없습니다.')
    scored=[]
    for cfg,vals in records.items():
        yt=[x[0] for x in vals]; yp=[x[1] for x in vals]
        scored.append((f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0),accuracy_score(yt,yp),cfg))
    scored.sort(key=lambda x:(x[0],x[1]),reverse=True)
    return scored[0][2],scored[0][0],scored[0][1],used,len(records[next(iter(records))])
def fold_path(f): return WORK/f'fold_{f:02d}.csv'
for fold,(tr,te) in enumerate(splits,1):
    if not FOLD_START<=fold<=FOLD_END: continue
    if fold_path(fold).exists(): print('Fold',fold,'완료 - skip'); continue
    train_labels=set(eval_df.iloc[tr].hs4.astype(str))
    test_labels=set(eval_df.iloc[te].hs4.astype(str))
    if not test_labels.issubset(train_labels): raise ValueError(f'Fold {fold}: 학습 데이터에 없는 HS4')
    selected={}
    for v in ['B','C','D']: selected[v]=tune_variant(tr,v)
    rows=[]
    for q in te:
        base=baseline_rank(tr,q)
        configs={'A':None,'B':selected['B'][0],'C':selected['C'][0],'D':selected['D'][0]}
        for v in ['A','B','C','D']:
            cfg=configs[v]
            rank=base if v=='A' else None
            if v!='A':
                if v=='B':
                    m,w=cfg; rank=rrf_ablation(base,direct_rank_cache[q][m],None,w,0)
                elif v=='C':
                    m,w=cfg; rank=rrf_ablation(base,None,exclusion_rank_cache[q][m],0,w)
                else:
                    md,me,wd,we=cfg; rank=rrf_ablation(base,direct_rank_cache[q][md],exclusion_rank_cache[q][me],wd,we)
            rows.append({'row_index':int(q),'fold':fold,'heldout_family':eval_df.iloc[q].product_family,'true_hs4':str(eval_df.iloc[q].hs4),'model':{'A':'Baseline_34_RRF','B':'Direct_Only','C':'Exclusion_Only','D':'Direct_Plus_Exclusion'}[v],'predicted_hs4':str(rank.index[0]),'top3_hs4':'|'.join(rank.index[:3].astype(str)),'selected_config':str(cfg),'inner_macro_f1':np.nan if v=='A' else selected[v][1],'inner_accuracy':np.nan if v=='A' else selected[v][2],'inner_valid_folds':np.nan if v=='A' else selected[v][3],'inner_validation_rows':np.nan if v=='A' else selected[v][4]})
    pd.DataFrame(rows).to_csv(fold_path(fold),index=False,encoding='utf-8-sig')
    print(f'Fold {fold}/{len(splits)} 저장 | B={selected["B"][0]} | C={selected["C"][0]} | D={selected["D"][0]}') 
print('현재 실행 구간 완료')

## 6. 진행률 및 중간 백업

In [ ]:
done=sorted(int(p.stem.split('_')[1]) for p in WORK.glob('fold_*.csv'))
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
BACKUP='/content/37_fold_backup.zip'
with zipfile.ZipFile(BACKUP,'w',zipfile.ZIP_DEFLATED) as z:
    for p in WORK.glob('fold_*.csv'): z.write(p,p.name)
    if CACHE.exists(): z.write(CACHE,CACHE.name)
from google.colab import files
files.download(BACKUP)

## 7. 중간 백업 복구 (새 런타임에서만 실행)

In [ ]:
from google.colab import files
uploaded=files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z: z.extractall(WORK)
        print('복구 완료:',name)

## 8. 통합 평가 및 결과 ZIP

In [ ]:
files_list=sorted(WORK.glob('fold_*.csv'))
done=sorted(int(p.stem.split('_')[1]) for p in files_list)
missing=sorted(set(range(1,len(splits)+1))-set(done))
print(f'완료 Fold: {len(done)}/{len(splits)} | 누락: {missing}')
if missing: raise ValueError('미완료 Fold가 있습니다.')
pred=pd.concat([pd.read_csv(p) for p in files_list],ignore_index=True)
if pred.duplicated(['row_index','model']).any(): raise ValueError('중복 예측 존재')
def metric(g):
    yt=g.true_hs4.astype(str); yp=g.predicted_hs4.astype(str)
    return pd.Series({'n':len(g),'HS4_Macro_F1':f1_score(yt,yp,labels=hs4_labels,average='macro',zero_division=0),'HS4_Accuracy':accuracy_score(yt,yp)})
summary=pred.groupby('model').apply(metric,include_groups=False).reset_index().sort_values(['HS4_Macro_F1','HS4_Accuracy'],ascending=False)
class_rows=[]
for model,g in pred.groupby('model'):
    p,r,f,s=precision_recall_fscore_support(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels,zero_division=0)
    for c,pv,rv,fv,sv in zip(hs4_labels,p,r,f,s): class_rows.append({'model':model,'HS4':c,'support':int(sv),'precision':pv,'recall':rv,'F1':fv})
class_metrics=pd.DataFrame(class_rows)
config_summary=pred[pred.model!='Baseline_34_RRF'].groupby(['model','fold','selected_config','inner_valid_folds','inner_validation_rows'],dropna=False).size().reset_index(name='outer_predictions')
display(summary.round(4))
display(class_metrics.sort_values(['model','HS4']).round(4))
display(config_summary)
OUT=WORK/'final'; OUT.mkdir(exist_ok=True)
pred.to_csv(OUT/'37_oof_predictions.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'37_model_comparison.csv',index=False,encoding='utf-8-sig')
class_metrics.to_csv(OUT/'37_hs4_class_metrics.csv',index=False,encoding='utf-8-sig')
config_summary.to_csv(OUT/'37_selected_configs_by_fold.csv',index=False,encoding='utf-8-sig')
for model,g in pred.groupby('model'):
    pd.DataFrame(confusion_matrix(g.true_hs4.astype(str),g.predicted_hs4.astype(str),labels=hs4_labels),index=hs4_labels,columns=hs4_labels).to_csv(OUT/f'37_confusion_{model}.csv',encoding='utf-8-sig')
evidence=pd.concat([direct_docs.assign(channel='direct'),exclusion_docs.assign(channel='exclusion')],ignore_index=True)
evidence[['knowledge_id','candidate_hs4','channel','rule_type','source_page_or_line','document_text']].to_csv(OUT/'37_kb_evidence_inventory.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile('/content/37_hs4_explanatory_note_ablation_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob('*.csv'): z.write(p,p.name)
print('저장 완료: /content/37_hs4_explanatory_note_ablation_results.zip')

## 결과 해석 기준

- D가 A보다 높으면 두 해설서 채널을 함께 활용하는 것이 유효한지 뒷받침합니다.
- B가 A보다 높고 C가 그렇지 않다면 직접 근거 채널의 기여가 더 클 수 있습니다.
- C가 A보다 높고 B가 그렇지 않다면 제외 근거 채널의 기여가 더 클 수 있습니다.
- B와 C가 각각 개선되지만 D가 더 낮다면 두 채널의 결합 또는 가중치 상호작용을 점검해야 합니다.
- 3306·3401은 표본이 각각 5건이므로 클래스별 F1 변화는 매우 불안정할 수 있습니다.
- Outer LOPO Macro-F1을 주 판단 기준으로 삼고 Accuracy와 클래스별 Recall을 함께 확인합니다.